# Proyecto 3 — Mejorando algoritmos con CUDA
## Aceleración en GPU de un detector de fraude con LSTM (PyTorch)

**Universidad del Valle — Computación Paralela**

**Integrantes (equipo de 3):**
- Pablo Cabrera — 231156
- _(integrante 2 — carné)_
- _(integrante 3 — carné)_

---

### Contexto

Este proyecto parte del **Laboratorio 3 — Secuencias: RNN y LSTM** (detección de fraude
transaccional del *Banco del Altiplano*), desarrollado previamente en el curso de Deep Learning con
**PyTorch**. Nos acogemos a la *Nota adicional* del enunciado, que permite acelerar **otro** laboratorio
previo en lugar del de ASL.

El objetivo es **trasladar la carga matemática de la red neuronal al GPU mediante CUDA** y medir la
mejora real de rendimiento, manteniendo la **preparación de datos en la CPU** (multinúcleo) y delegando
**exclusivamente el entrenamiento del modelo al GPU**, tal como exige la rúbrica.

> **Cómo correr este notebook:** en Google Colab, menú **Entorno de ejecución → Cambiar tipo de entorno
> de ejecución → Acelerador por hardware: GPU (T4)**. Luego *Entorno de ejecución → Ejecutar todo*.


---
## 0. Entorno y hardware

Primero verificamos qué GPU asignó Colab y que PyTorch reconozca CUDA. La captura de `nvidia-smi`
de esta celda es una de las evidencias que pide el informe (30 pts de *Evidencia de Ejecución*).

In [ ]:
# Evidencia de hardware GPU asignado por Colab (capturar pantalla de esta salida)
!nvidia-smi

In [ ]:
import time, platform, multiprocessing
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("Python     :", platform.python_version())
print("PyTorch    :", torch.__version__)
print("NumPy      :", np.__version__)
print("CUDA build :", torch.version.cuda)
print("¿CUDA disponible?:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo activo:", DEVICE)

if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
    print("cuDNN      :", torch.backends.cudnn.version())
else:
    print("ADVERTENCIA: no hay GPU. Activa el runtime GPU en Colab para ver el speedup real.")

print("Núcleos lógicos de CPU:", multiprocessing.cpu_count())

---
## 1. Análisis de la solución secuencial existente

La solución original entrena una **LSTM** que lee la *secuencia* de movimientos de una tarjeta y
decide si hay fraude. Todo el cómputo del laboratorio original corre en **CPU**.

### 1.1 Principales partes del programa

| No. | Operación identificada | Descripción de su funcionamiento |
|----:|------------------------|----------------------------------|
| 1 | **Generación / carga de datos** | Crea `N` secuencias de `T` pasos (monto normalizado). Los fraudes llevan una escalada al final; los normales, los mismos montos desordenados. Es trabajo de **CPU** (NumPy, lógica de arreglos). |
| 2 | **Empaquetado en tensores y `DataLoader`** | Convierte los arreglos a tensores `(batch, T, 1)` y los sirve en lotes barajados. Manejo de memoria y E/S → **CPU**. |
| 3 | **Definición del modelo** (`DetectorFraude`) | Una capa `nn.LSTM(input_size=1, hidden_size=H)` + una `nn.Linear(H, 1)` que produce el logit por secuencia. |
| 4 | **Paso forward** | Multiplicaciones matriciales de las 4 compuertas de la LSTM a lo largo de los `T` pasos + capa lineal. **Matemática densa** → candidata a GPU. |
| 5 | **Cálculo de la pérdida** (`BCEWithLogitsLoss`) | Compara el logit con la etiqueta. |
| 6 | **Backward (retropropagación)** | Calcula los gradientes de todos los pesos. Es la operación **más costosa**; vuelve a ser matemática matricial densa → candidata a GPU. |
| 7 | **Paso del optimizador** (`Adam`) | Actualiza los pesos con los gradientes. Operaciones elemento a elemento sobre los parámetros → GPU. |
| 8 | **Evaluación / métrica** | Forward sin gradientes sobre validación y cálculo de exactitud. |

### 1.2 Operaciones que se ejecutan repetidamente

| Operación repetitiva | Datos que procesa | Cantidad de repeticiones |
|----------------------|-------------------|--------------------------|
| Forward + backward + `step` de un lote | Un lote de `batch` secuencias `(batch, T, 1)` | `épocas × ⌈N/batch⌉` veces por entrenamiento |
| Recurrencia interna de la LSTM (4 compuertas por paso) | El estado `(h, c)` de tamaño `H` | `T` pasos por cada secuencia del lote |
| Entrenamiento completo de un modelo | Todo el dataset | 1 por modelo — pero el *barrido* del experimento entrena **decenas** de modelos |

👉 **Las operaciones 4, 6 y 7 son matemática matricial densa y se repiten millones de veces.** Ese es
exactamente el tipo de trabajo que una GPU ejecuta en paralelo mucho más rápido que una CPU.

### 1.3 Código original (versión secuencial de referencia)

A continuación, el código del laboratorio tal como se desarrolló. Está parametrizado para poder
elegir el dispositivo, pero **por defecto corre en CPU** (comportamiento original).

In [ ]:
# ---- Generación de datos (CPU / NumPy) -------------------------------------
# El orden de los movimientos es la ÚNICA señal: ambas clases tienen los mismos
# montos, solo cambia dónde cae la escalada. Por eso las variables agregadas
# (promedio, máximo, desviación) no distinguen las clases y hace falta la LSTM.
def generar_datos(n, largo, semilla=7):
    r = np.random.default_rng(semilla)
    X = r.uniform(0.05, 0.45, size=(n, largo))
    y = np.zeros(n)
    idx_fraude = r.choice(n, size=n // 2, replace=False)
    escalada = np.linspace(0.5, 0.95, 6)
    set_fraude = set(idx_fraude.tolist())
    for i in idx_fraude:
        X[i, -6:] = escalada + r.normal(0, 0.03, size=6)   # escalada al final
        y[i] = 1.0
    for i in range(n):
        if i in set_fraude:
            continue
        posiciones = r.choice(largo, size=6, replace=False)  # mismos montos, dispersos
        X[i, posiciones] = escalada + r.normal(0, 0.03, size=6)
    return X.astype(np.float32), y.astype(np.float32)


# ---- Modelo (idéntico al del laboratorio) ----------------------------------
class DetectorFraude(nn.Module):
    def __init__(self, oculto=32, entradas=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=entradas, hidden_size=oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)

    def forward(self, x):
        _, (h_n, _c_n) = self.lstm(x)   # h_n[-1] resume TODA la secuencia
        return self.salida(h_n[-1])


# ---- Construcción del DataLoader (CPU) -------------------------------------
def construir_loader(X, y, batch_size, num_workers=0, pin_memory=False):
    ds = TensorDataset(torch.from_numpy(X).unsqueeze(-1), torch.from_numpy(y).unsqueeze(-1))
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=num_workers, pin_memory=pin_memory)


print("Funciones base definidas: generar_datos, DetectorFraude, construir_loader")

---
## 2. Medición del tiempo de ejecución secuencial (CPU)

Definimos una **función de entrenamiento cronometrada** que separa el tiempo en dos fases:

- **`t_transfer`** — mover cada lote al dispositivo (`.to(device)`). En CPU es ~0; en GPU es el costo
  de la transferencia **CPU↔GPU** por el bus PCIe.
- **`t_compute`** — forward + backward + `step` (la matemática de la red).

Para medir bien la GPU usamos `torch.cuda.synchronize()` (las llamadas a CUDA son asíncronas; sin
sincronizar, el reloj mediría solo el *lanzamiento* de los kernels, no su ejecución).

In [ ]:
def entrenar_cronometrado(device, cfg, semilla=0, lr=1e-2, warmup=False):
    '''Entrena un DetectorFraude y devuelve tiempos por fase (en segundos).

    cfg = dict(N, T, hidden, batch, epocas)
    '''
    torch.manual_seed(semilla)
    es_cuda = (device.type == "cuda")

    # --- FASE CPU: preparación de datos (no se delega al GPU) ---
    X, y = generar_datos(cfg["N"], cfg["T"], semilla=7)
    loader = construir_loader(X, y, cfg["batch"],
                              num_workers=0, pin_memory=es_cuda)

    modelo = DetectorFraude(oculto=cfg["hidden"]).to(device)
    criterio = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)

    if es_cuda:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    t_transfer = 0.0
    t0 = time.perf_counter()
    perdida_final = 0.0
    for _ in range(cfg["epocas"]):
        for xb, yb in loader:
            # --- transferencia CPU -> GPU (medida aparte) ---
            if es_cuda:
                torch.cuda.synchronize(); ta = time.perf_counter()
            xb = xb.to(device, non_blocking=es_cuda)
            yb = yb.to(device, non_blocking=es_cuda)
            if es_cuda:
                torch.cuda.synchronize(); t_transfer += time.perf_counter() - ta
            # --- cómputo de la red (GPU/CUDA) ---
            opt.zero_grad()
            perdida = criterio(modelo(xb), yb)
            perdida.backward()
            opt.step()
            perdida_final = perdida.item()
    if es_cuda:
        torch.cuda.synchronize()
    t_total = time.perf_counter() - t0

    mem = (torch.cuda.max_memory_allocated() / 1e6) if es_cuda else 0.0
    return {
        "t_total": t_total,
        "t_transfer": t_transfer,
        "t_compute": t_total - t_transfer,
        "perdida": perdida_final,
        "mem_MB": mem,
    }


def promedio_corridas(device, cfg, repeticiones=3, warmup=True):
    '''Corre varias veces bajo las mismas condiciones y promedia (descarta warmup).'''
    if warmup:   # 1 corrida de calentamiento (compila kernels CUDA, llena cachés)
        entrenar_cronometrado(device, cfg, semilla=99)
    reg = [entrenar_cronometrado(device, cfg, semilla=s) for s in range(repeticiones)]
    prom = {k: float(np.mean([r[k] for r in reg])) for k in ("t_total", "t_transfer", "t_compute", "mem_MB")}
    prom["t_total_std"] = float(np.std([r["t_total"] for r in reg]))
    prom["repeticiones"] = repeticiones
    return prom


print("Harness de cronometraje listo.")

### 2.1 Configuración *pequeña* (la del laboratorio original)

Es el tamaño original: `N=2000`, `T=30`, `hidden=32`. Es tan pequeño que **la CPU lo resuelve en
menos de un minuto**. Lo medimos para tenerlo como línea base y, más adelante, mostrar que a este
tamaño el GPU **no** ofrece ventaja (el costo de transferencia y lanzamiento de kernels domina).

In [ ]:
CFG_PEQUENA = dict(N=2000, T=30, hidden=32, batch=64, epocas=6)

print("Entrenando configuración PEQUEÑA en CPU...")
cpu_peq = promedio_corridas(torch.device("cpu"), CFG_PEQUENA, repeticiones=3)
print(f"  CPU  t_total = {cpu_peq['t_total']:.3f} s  (±{cpu_peq['t_total_std']:.3f})")

---
## 3. Identificación de oportunidades de paralelización

Del análisis de la sección 1, las operaciones se dividen en dos grupos según dónde conviene ejecutarlas:

**Se quedan en la CPU (manipulación de datos / control):**
- Generación de las secuencias (`generar_datos`), lógica de arreglos NumPy.
- Empaquetado en tensores y el barajado por lotes del `DataLoader` (puede usar **varios núcleos** con
  `num_workers>0`).
- Control del bucle de entrenamiento y registro de métricas.

**Se delegan al GPU con CUDA (matemática de la red — operaciones 4, 6, 7):**
- **Forward de la LSTM**: en cada paso de tiempo se calculan 4 compuertas, cada una es un producto
  matriz-vector sobre el lote completo. Con `batch × hidden` elementos, hay miles de multiplicaciones
  independientes que la GPU hace **en paralelo**.
- **Backward**: el gradiente de esas mismas matrices; es la fase más pesada.
- **Paso de Adam**: actualización elemento a elemento de todos los parámetros.

**Justificación:** la GPU tiene miles de núcleos pensados para operaciones SIMD (la misma operación
sobre muchos datos). Una multiplicación de matrices de `batch=512 × hidden=256` son >130 000
multiplicaciones-acumulaciones *independientes* por compuerta y por paso: el caso ideal para la GPU.
En cambio, generar los datos es lógica secuencial con ramas y acceso irregular a memoria — eso la CPU
lo hace mejor, y además no tiene sentido pagar el costo de subirlo a la GPU para una operación que se
hace una sola vez.

---
## 4. Diseño del algoritmo paralelo (CUDA)

No escribimos kernels CUDA a mano: usamos **PyTorch**, que internamente llama a **cuDNN/cuBLAS**, las
librerías de NVIDIA que ya traen kernels de LSTM altamente optimizados. El diseño consiste en **decidir
qué datos viven en el GPU y cómo se reparte el trabajo** en la jerarquía CUDA de *hilos → bloques → grid*:

**Jerarquía CUDA aplicada a nuestro entrenamiento:**
- **Hilo (thread):** calcula **un elemento** del resultado — p. ej. una entrada del vector de estado
  `h` para una secuencia del lote (una multiplicación-acumulación).
- **Bloque (block):** un grupo de hilos que cooperan y comparten *memoria compartida*. cuBLAS agrupa la
  multiplicación de una **sub-matriz (tile)** de la operación `W·x` en un bloque, reutilizando datos en
  memoria rápida.
- **Grid:** todos los bloques necesarios para cubrir la multiplicación completa de las 4 compuertas
  sobre las `batch` secuencias. El grid se relanza (o encadena) en cada uno de los `T` pasos de tiempo,
  porque la recurrencia es secuencial en el tiempo (`h_t` depende de `h_{t-1}`), **pero dentro de cada
  paso las `batch × hidden` cuentas son totalmente paralelas**.

**Reparto concreto del trabajo:**

| Dimensión del problema | Cómo se paraleliza en CUDA |
|---|---|
| `batch` (secuencias del lote) | Independientes entre sí → se reparten entre bloques/hilos. |
| `hidden` (tamaño del estado) | Cada componente de `h` y `c` es un hilo distinto. |
| `T` (pasos de tiempo) | **Secuencial** (la recurrencia obliga el orden), pero cuDNN fusiona las 4 compuertas en un solo kernel por paso para minimizar lanzamientos. |
| Parámetros (backward + Adam) | Gradiente y actualización elemento a elemento → un hilo por parámetro. |

**Flujo de datos CPU↔GPU (lo que mediremos como `t_transfer`):**
1. CPU genera el lote y lo deja en memoria *pinned* (`pin_memory=True`).
2. Se copia al GPU por PCIe (`xb.to('cuda')`).
3. El GPU ejecuta forward+backward+step sin volver a la CPU.
4. Solo regresa a la CPU un escalar (la pérdida) para registro.

Minimizar los viajes CPU↔GPU es clave: por eso **solo** se transfiere el lote de entrada, no resultados
intermedios.

---
## 5. Implementación con CUDA Python y medición paralela

Usamos **la misma** función `entrenar_cronometrado`: lo único que cambia es `device='cuda'`. Esa es la
esencia del enfoque — *la preparación de datos no cambia (sigue en CPU), solo el modelo y los lotes
viajan al GPU*.

### 5.1 Configuración *escalada*

Para que el GPU muestre su ventaja, escalamos el problema a un tamaño realista (más datos, secuencias
más largas y un estado oculto grande). Esto hace que la **matemática** domine sobre la transferencia.

> Si el entrenamiento en CPU tarda demasiado en tu Colab, baja `N` o `epocas`. Si quieres un speedup
> aún mayor, sube `hidden` a 512.

In [ ]:
# Dimensionada para Colab free (CPU de 2 núcleos). Para un speedup aún mayor,
# sube N, hidden (p. ej. 512) o epocas si tu entorno lo permite.
CFG_ESCALADA = dict(N=6000, T=50, hidden=256, batch=256, epocas=3)

print("CONFIG ESCALADA:", CFG_ESCALADA)
print()

print("Entrenando en CPU (esto es lo lento)...")
cpu_esc = promedio_corridas(torch.device("cpu"), CFG_ESCALADA, repeticiones=3, warmup=False)
print(f"  CPU  t_total = {cpu_esc['t_total']:.3f} s  (±{cpu_esc['t_total_std']:.3f})")
print()

if torch.cuda.is_available():
    print("Entrenando en GPU (CUDA)...")
    gpu_esc = promedio_corridas(torch.device("cuda"), CFG_ESCALADA, repeticiones=3, warmup=True)
    print(f"  GPU  t_total   = {gpu_esc['t_total']:.3f} s  (±{gpu_esc['t_total_std']:.3f})")
    print(f"       t_compute = {gpu_esc['t_compute']:.3f} s   (matemática de la red)")
    print(f"       t_transfer= {gpu_esc['t_transfer']:.3f} s   (CPU<->GPU por PCIe)")
    print(f"       memoria GPU pico = {gpu_esc['mem_MB']:.1f} MB")
else:
    gpu_esc = None
    print("Sin GPU: activa el runtime GPU en Colab para medir CUDA.")

### 5.2 Evidencia de uso del GPU (`nvidia-smi` + memoria)

Ejecuta esta celda **justo después** del entrenamiento en GPU para capturar el uso de memoria y de los
núcleos. Para una captura *durante* la carga, corre un entrenamiento largo y abre otra celda con
`!nvidia-smi` mientras entrena (o usa `nvidia-smi -l 1`).

In [ ]:
if torch.cuda.is_available():
    print("Memoria GPU reservada ahora:", torch.cuda.memory_reserved() / 1e6, "MB")
    print("Memoria GPU pico durante el entrenamiento:", torch.cuda.max_memory_allocated() / 1e6, "MB")
    print()
!nvidia-smi

---
## 6. Benchmark escalado: ¿cuándo gana la GPU?

Entrenamos el **mismo** modelo variando el tamaño del estado oculto (`hidden`) y comparamos CPU vs GPU.
Esto responde directamente a la pregunta de la rúbrica *"qué factores justifican o limitan el uso de
CUDA"*: a problema pequeño la GPU pierde por overhead; a medida que crece, el speedup se dispara.

In [ ]:
TAMANOS = [16, 32, 64, 128, 256]
BASE = dict(N=2000, T=30, batch=256, epocas=2)

filas = []
for h in TAMANOS:
    cfg = dict(BASE, hidden=h)
    c = promedio_corridas(torch.device("cpu"), cfg, repeticiones=2, warmup=False)["t_total"]
    if torch.cuda.is_available():
        g = promedio_corridas(torch.device("cuda"), cfg, repeticiones=2, warmup=True)["t_total"]
        speedup = c / g
    else:
        g, speedup = float("nan"), float("nan")
    filas.append((h, c, g, speedup))
    print(f"hidden={h:4d} | CPU {c:7.3f}s | GPU {g:7.3f}s | speedup {speedup:5.2f}x")

import pandas as pd
tabla_sweep = pd.DataFrame(filas, columns=["hidden", "CPU_s", "GPU_s", "speedup"])
tabla_sweep

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(tabla_sweep["hidden"], tabla_sweep["CPU_s"], "o-", label="CPU")
if torch.cuda.is_available():
    ax1.plot(tabla_sweep["hidden"], tabla_sweep["GPU_s"], "s-", label="GPU (CUDA)")
ax1.set_xlabel("hidden_size"); ax1.set_ylabel("tiempo (s)")
ax1.set_title("Tiempo de entrenamiento vs tamaño del modelo"); ax1.legend(); ax1.grid(True, alpha=.3)

if torch.cuda.is_available():
    ax2.plot(tabla_sweep["hidden"], tabla_sweep["speedup"], "d-", color="green")
    ax2.axhline(1.0, color="red", ls="--", label="speedup = 1 (empate)")
    ax2.set_xlabel("hidden_size"); ax2.set_ylabel("speedup (CPU/GPU)")
    ax2.set_title("Speedup del GPU vs tamaño del modelo"); ax2.legend(); ax2.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

---
## 7. Análisis y conclusiones

### 7.1 Tabla comparativa de tiempos y speedup por fase

In [ ]:
import pandas as pd

tabla = pd.DataFrame([
    {
        "Config": "Escalada",
        "CPU total (s)": round(cpu_esc["t_total"], 3),
        "GPU total (s)": round(gpu_esc["t_total"], 3) if gpu_esc else None,
        "GPU cómputo (s)": round(gpu_esc["t_compute"], 3) if gpu_esc else None,
        "GPU transfer. (s)": round(gpu_esc["t_transfer"], 3) if gpu_esc else None,
        "Speedup total (CPU/GPU)": round(cpu_esc["t_total"] / gpu_esc["t_total"], 2) if gpu_esc else None,
    }
])
tabla

### 7.2 Medición de la configuración pequeña también en GPU

Para mostrar el contraste (a tamaño pequeño el GPU no gana) medimos la config pequeña en ambos.

In [ ]:
if torch.cuda.is_available():
    gpu_peq = promedio_corridas(torch.device("cuda"), CFG_PEQUENA, repeticiones=3, warmup=True)
    sp_peq = cpu_peq["t_total"] / gpu_peq["t_total"]
    print(f"PEQUEÑA  CPU {cpu_peq['t_total']:.3f}s | GPU {gpu_peq['t_total']:.3f}s | speedup {sp_peq:.2f}x")
    print(f"ESCALADA CPU {cpu_esc['t_total']:.3f}s | GPU {gpu_esc['t_total']:.3f}s | speedup {cpu_esc['t_total']/gpu_esc['t_total']:.2f}x")
    print()
    resumen = pd.DataFrame([
        {"Config": "Pequeña",  "CPU (s)": round(cpu_peq['t_total'],3), "GPU (s)": round(gpu_peq['t_total'],3), "Speedup": round(sp_peq,2)},
        {"Config": "Escalada", "CPU (s)": round(cpu_esc['t_total'],3), "GPU (s)": round(gpu_esc['t_total'],3), "Speedup": round(cpu_esc['t_total']/gpu_esc['t_total'],2)},
    ])
    display(resumen)

    fig, ax = plt.subplots(figsize=(7,4))
    x = np.arange(2); w = 0.35
    ax.bar(x-w/2, [cpu_peq['t_total'], cpu_esc['t_total']], w, label="CPU")
    ax.bar(x+w/2, [gpu_peq['t_total'], gpu_esc['t_total']], w, label="GPU (CUDA)")
    ax.set_xticks(x); ax.set_xticklabels(["Pequeña", "Escalada"])
    ax.set_ylabel("tiempo total (s)"); ax.set_title("CPU vs GPU por configuración")
    ax.legend(); ax.grid(True, axis="y", alpha=.3)
    for i,(c,g) in enumerate([(cpu_peq['t_total'],gpu_peq['t_total']),(cpu_esc['t_total'],gpu_esc['t_total'])]):
        ax.text(i, max(c,g)*1.02, f"{c/g:.1f}x", ha="center", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("Sin GPU disponible.")

### 7.3 Discusión técnica (respuestas del informe)

> **Nota:** redacta estas respuestas en el informe PDF con los *números reales* que te dio tu corrida
> en Colab. Aquí queda el argumento técnico base.

**¿Por qué procesar los datos/archivos con la CPU y delegar solo la matemática de la red al GPU?**
La preparación de datos es lógica secuencial con ramas, acceso irregular a memoria y operaciones que se
hacen una sola vez; la CPU, con pocos núcleos pero muy rápidos y buena predicción de saltos, es más
eficiente en eso, y puede repartir la carga de E/S entre varios núcleos (`num_workers`). La matemática de
la red, en cambio, es la misma operación (multiplicar-acumular) sobre miles de datos independientes: ahí
los miles de núcleos SIMD del GPU rinden mucho más. Subir los datos al GPU para generarlos no compensa:
se pagaría la transferencia por una operación que no se beneficia del paralelismo masivo.

**¿Cómo permitió CUDA reducir el tiempo?**
Al mover el modelo y los lotes al GPU, PyTorch ejecuta el forward/backward con kernels de **cuDNN/cuBLAS**
que paralelizan las multiplicaciones de las compuertas de la LSTM sobre todo el lote a la vez. El tiempo
de la fase pesada (operaciones 4, 6, 7) se reduce varias veces; lo medimos en `t_compute`.

**¿Qué factores justifican o limitan el uso de CUDA?**
- *Justifican:* modelos grandes (`hidden` alto), lotes grandes, muchos datos y muchas repeticiones
  (como el barrido de modelos). Cuanto más domina la matemática, mayor el speedup.
- *Limitan:* el costo fijo de transferencia **CPU↔GPU** (`t_transfer`) y de lanzar kernels; la
  **recurrencia** de la LSTM, que obliga a procesar los `T` pasos en orden; y problemas pequeños, donde
  ese overhead supera la ganancia (por eso la config pequeña puede no mostrar speedup, o incluso ser más
  lenta en GPU).

**Librerías/funciones que permitieron usar el GPU:**
`torch.cuda`, `tensor.to('cuda')`, `nn.LSTM`/`nn.Linear` (que despachan a **cuDNN** y **cuBLAS**),
`pin_memory=True` y `non_blocking=True` para solapar transferencia, y `torch.cuda.synchronize()` /
`torch.cuda.Event` para medir correctamente.

---
## 8. Conclusión

- El pipeline quedó **estrictamente separado**: generación/carga de datos en **CPU** (NumPy + `DataLoader`)
  y entrenamiento de la LSTM en **GPU** vía CUDA.
- A tamaño pequeño (el original), el GPU **no** aporta ventaja: domina el overhead de transferencia.
- Al **escalar** el problema, el speedup del GPU crece de forma clara (ver tabla y gráficos).
- La **recurrencia temporal** de la LSTM es el principal límite al paralelismo; aun así, la paralelización
  dentro de cada paso (sobre `batch × hidden`) basta para que el GPU gane holgadamente en modelos grandes.

_Rellena las cifras finales en el informe PDF con los resultados de tu corrida en Colab._